In [16]:
# define the Sequential Cognitive Attention Block (SCAB)
import torch.nn as nn
class ChannelGate(nn.Module):
    def __init__(self, gate_channels, reduction_ratio=16):
        super(ChannelGate, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.Linear1 = nn.Conv2d(gate_channels, gate_channels // reduction_ratio, 1, bias=False)
        self.BatchNorm1d = nn.BatchNorm1d(gate_channels // reduction_ratio)
        self.ReLU = nn.ReLU()
        self.Linear2 = nn.Conv2d(gate_channels // reduction_ratio, gate_channels, 1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.Linear2(self.ReLU(self.Linear1(self.avg_pool(x))))
        max_out = self.Linear2(self.ReLU(self.Linear1(self.max_pool(x))))
        out = avg_out + max_out
        return self.sigmoid(out)

class SpatialGate(nn.Module):
    def __init__(self, kernel_size=3, number_of_dilation=1, dilation_value=2):# dilation_value=dilation_rate
        super(SpatialGate, self).__init__()
        # the receptive field of dilated convolution with a kernel size of 3 × 3 and a dilation value of 2 is equal to 5 × 5.
        self.conv = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size=3, padding=dilation_value, dilation=dilation_value),
            nn.BatchNorm2d(1),
            nn.ReLU()
            )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        x = self.conv(x)
        return self.sigmoid(x)

class SCAB(nn.Module):
    def __init__(self, gate_channels):
        super(SCAB, self).__init__()

        self.channel_att = ChannelGate(gate_channels)
        self.spatial_att = SpatialGate()

    def forward(self, x):

        channel_att_map = x * (self.channel_att(x))
        out = x + channel_att_map * (self.spatial_att(channel_att_map))
        return out